In [11]:
from iapws import IAPWS97

def calculate_ideal_dryness(P_ms, T_ms, P_cond):
    """
    根据主蒸汽入口状态和凝汽器压力，计算汽轮机的理想排汽干度。

    参数 (Parameters):
    ----------
    P_ms : float
        主蒸汽压力 (Main Steam Pressure)，单位：MPa (兆帕)。
    T_ms : float
        主蒸汽温度 (Main Steam Temperature)，单位：°C (摄氏度)。
    P_cond : float
        凝汽器压力 (Condenser Pressure)，单位：MPa (兆帕)。
        注意：通常测量的是真空度(kPa)，需要转换为绝对压力(MPa)。
        绝对压力(MPa) = (大气压(kPa) - 真空度(kPa)) / 1000

    返回 (Returns):
    -------
    float
        理想排汽干度 (Ideal Exhaust Steam Dryness)。
        如果排汽处于过热区，返回值会大于1。
        如果输入参数无效或处于不支持的区域，可能返回None或抛出异常。
    """
    try:
        # 1. 获取主蒸汽入口状态点 (点1)
        # IAPWS97需要使用开尔文(K)作为温度单位，所以需要转换
        T_ms_K = T_ms + 273.15
        inlet_steam = IAPWS97(P=P_ms, T=T_ms_K)
        
        # 2. 从入口点获取比熵(s)，这是理想膨胀过程中的不变量
        s_ms = inlet_steam.s  # 单位: kJ/(kg*K)
        
        # 3. 获取排汽压力下的饱和状态点 (点2)
        # 注意：这里我们传入压力和干度x=0来获取饱和水点
        outlet_sat_liquid = IAPWS97(P=P_cond, x=0)
        s_f = outlet_sat_liquid.s  # 获取饱和水比熵 s_f
        
        # 传入压力和干度x=1来获取干饱和蒸汽点
        outlet_sat_vapor = IAPWS97(P=P_cond, x=1)
        s_g = outlet_sat_vapor.s  # 获取干饱和蒸汽比熵 s_g

        # 4. 判断排汽状态并计算理想干度
        if s_ms < s_f:
            # 这种情况物理上很少见，意味着排汽是过冷水
            # 但为了代码完整性，我们处理它
            print(f"警告: 理想排汽点 (s={s_ms:.4f}) 处于过冷水区 (s_f={s_f:.4f})。")
            # 在这种情况下，干度没有物理意义，通常视为0
            return 0.0
        elif s_ms > s_g:
            # 理想排汽点处于过热蒸汽区
            print(f"信息: 理想排汽点 (s={s_ms:.4f}) 处于过热蒸汽区 (s_g={s_g:.4f})。")
            # 在过热区，干度 > 1。我们可以直接返回计算值，或简单标记为1以上。
            # 为了数值分析，返回实际计算值更有用。
            x_s = (s_ms - s_f) / (s_g - s_f)
            return x_s
        else:
            # 理想排汽点处于湿蒸汽区 (s_f <= s_ms <= s_g)
            print(f"信息: 理想排汽点 (s={s_ms:.4f}) 处于湿蒸汽区。")
            x_s = (s_ms - s_f) / (s_g - s_f)
            return x_s

    except Exception as e:
        print(f"计算过程中发生错误: {e}")
        print(f"请检查输入参数是否在有效范围内: P_ms={P_ms}, T_ms={T_ms}, P_cond={P_cond}")
        return None

# --- 使用示例 ---
if __name__ == '__main__':
    # 示例1：一个典型的高参数机组工况
    main_steam_pressure = 13  # MPa
    main_steam_temperature = 538  # °C
    condenser_pressure = (101.325+73.7)/1000 # MPa (对应约 2.7 kPa的绝对压力，真空度很高)
    
    ideal_dryness = calculate_ideal_dryness(main_steam_pressure, main_steam_temperature, condenser_pressure)
    
    if ideal_dryness is not None:
        print(f"\n计算结果:")
        print(f"主蒸汽压力: {main_steam_pressure} MPa")
        print(f"主蒸汽温度: {main_steam_temperature} °C")
        print(f"凝汽器压力: {condenser_pressure} MPa")
        print(f"理想排汽干度 x_s: {ideal_dryness:.4f}")
        
        # 应用安全约束检查
        safety_threshold = 0.88
        if ideal_dryness >= safety_threshold:
            print(f"状态: 安全 (理想干度 >= {safety_threshold})")
        else:
            print(f"状态: 危险 (理想干度 < {safety_threshold})")

    print("\n" + "="*30 + "\n")

    # 示例2：一个可能导致排汽过热的工况 (例如真空较低)
    condenser_pressure_low_vacuum = 0.01 # MPa (对应 10 kPa, 真空度较低)
    ideal_dryness_overheated = calculate_ideal_dryness(main_steam_pressure, main_steam_temperature, condenser_pressure_low_vacuum)
    
    if ideal_dryness_overheated is not None:
        print(f"计算结果 (低真空工况):")
        print(f"凝汽器压力: {condenser_pressure_low_vacuum} MPa")
        print(f"理想排汽干度 x_s: {ideal_dryness_overheated:.4f}")

信息: 理想排汽点 (s=6.5700) 处于湿蒸汽区。

计算结果:
主蒸汽压力: 13 MPa
主蒸汽温度: 538 °C
凝汽器压力: 0.17502500000000001 MPa
理想排汽干度 x_s: 0.8942
状态: 安全 (理想干度 >= 0.88)


信息: 理想排汽点 (s=6.5700) 处于湿蒸汽区。
计算结果 (低真空工况):
凝汽器压力: 0.01 MPa
理想排汽干度 x_s: 0.7895


In [ ]:
import pandas as pd
import numpy as np
from iapws import IAPWS97

def calculate_ideal_dryness(P_ms, T_ms, P_cond):
    """
    根据主蒸汽入口状态和凝汽器压力，计算汽轮机的理想排汽干度。
    此版本为批量处理优化，减少了打印信息。

    参数:
    P_ms : float - 主蒸汽压力 (MPa)
    T_ms : float - 主蒸汽温度 (°C)
    P_cond : float - 凝汽器绝对压力 (MPa)

    返回:
    float - 理想排汽干度 (x_s)，如果计算失败则返回 np.nan
    """
    # 检查输入是否为有效的数值类型
    if pd.isna(P_ms) or pd.isna(T_ms) or pd.isna(P_cond):
        return np.nan
        
    try:
        T_ms_K = T_ms + 273.15
        inlet_steam = IAPWS97(P=P_ms, T=T_ms_K)
        s_ms = inlet_steam.s
        
        # 检查P_cond是否在有效范围内
        # IAPWS97的饱和线最低压力约为 611.2 Pa (0.0006112 MPa)
        if P_cond < 0.000612:
             # print(f"Warning: Condenser pressure {P_cond:.6f} MPa is too low. Skipping.")
             return np.nan

        outlet_sat_liquid = IAPWS97(P=P_cond, x=0)
        s_f = outlet_sat_liquid.s
        
        outlet_sat_vapor = IAPWS97(P=P_cond, x=1)
        s_g = outlet_sat_vapor.s

        if s_g == s_f: # 避免除以零
            return np.nan
            
        x_s = (s_ms - s_f) / (s_g - s_f)
        return x_s

    except (ValueError, TypeError) as e:
        # 捕捉iapws可能抛出的各种错误
        # print(f"Calculation error for P_ms={P_ms}, T_ms={T_ms}, P_cond={P_cond}: {e}")
        return np.nan

def process_plant_data(filepath, atm_pressure_kpa=101.325):
    """
    读取电厂数据文件，计算关键参数，并计算理想排汽干度。

    参数:
    filepath : str - Excel文件的路径.
    atm_pressure_kpa : float - 当地大气压力，单位kPa. 默认为标准大气压.

    返回:
    pandas.DataFrame - 包含计算结果的新DataFrame.
    """
    print(f"正在读取文件: {filepath}...")
    try:
        df = pd.read_excel(filepath)
        print("文件读取成功！")
    except FileNotFoundError:
        print(f"错误: 文件 '{filepath}' 未找到。请确保文件名正确且文件与脚本在同一目录下。")
        return None
    
    # --- 1. 数据准备和清洗 ---
    print("正在准备数据...")
    
    # 计算均值，并处理可能只有单列存在的情况
    df['P_ms_MPa'] = df[['pt2502a', 'pt2502b']].mean(axis=1)
    df['T_ms_C'] = df['rdlc04']
    
    # 计算真空度均值
    vacuum_kpa = df[['pt2807a', 'pt2807b']].mean(axis=1)
    # vacuum_kpa=vacuum_kpa*0.94
    # --- 2. 关键步骤：将真空度转换为绝对压力 ---
    # 公式: 绝对压力(kPa) = 大气压(kPa) - 真空度(kPa)
    #       绝对压力(MPa) = (大气压(kPa) - 真空度(kPa)) / 1000
    print(f"使用大气压力 {atm_pressure_kpa} kPa 将真空度转换为绝对压力...")
    df['P_cond_MPa'] = (atm_pressure_kpa - vacuum_kpa) / 1000
    
    # --- 3. 批量计算干度 ---
    print("正在逐行计算理想排汽干度，这可能需要一些时间...")
    
    # 创建一个新的列 'ideal_dryness_xs'
    # 使用 .loc 避免 SettingWithCopyWarning
    df.loc[:, 'ideal_dryness_xs'] = df.apply(
        lambda row: calculate_ideal_dryness(row['P_ms_MPa'], row['T_ms_C'], row['P_cond_MPa']),
        axis=1
    )
    
    print("计算完成！")
    
    # --- 4. 结果分析 ---
    # 显示结果摘要
    print("\n--- 计算结果摘要 ---")
    print(df[['P_ms_MPa', 'T_ms_C', 'P_cond_MPa', 'ideal_dryness_xs']].describe())
    
    # 检查有多少计算失败了
    failed_calculations = df['ideal_dryness_xs'].isna().sum()
    total_rows = len(df)
    print(f"\n总行数: {total_rows}")
    print(f"成功计算行数: {total_rows - failed_calculations}")
    print(f"计算失败/跳过行数: {failed_calculations}")

    # 检查有多少排汽是过热的
    superheated_count = df[df['ideal_dryness_xs'] > 1].shape[0]
    if total_rows - failed_calculations > 0:
      percentage_superheated = (superheated_count / (total_rows - failed_calculations)) * 100
      print(f"过热蒸汽工况数量 (干度 > 1): {superheated_count} ({percentage_superheated:.2f}%)")
    
    return df


# --- 主程序 ---
if __name__ == '__main__':
    # 文件名，请确保与你的文件名一致
    filename = 'minute_Part234678-fulldata.xlsx'
    
    # **重要**: 如果你知道电厂当地的平均大气压，请修改这个值。
    # 标准大气压是 101.325 kPa。如果不知道，使用默认值即可。
    atmospheric_pressure = 101.325 
    
    # 执行处理流程
    result_df = process_plant_data(filename, atm_pressure_kpa=atmospheric_pressure)
    
    if result_df is not None:
        # 显示前5行和后5行结果
        print("\n--- 结果预览 (前5行) ---")
        print(result_df[['P_ms_MPa', 'T_ms_C', 'P_cond_MPa', 'ideal_dryness_xs']].head())
        print("\n--- 结果预览 (后5行) ---")
        print(result_df[['P_ms_MPa', 'T_ms_C', 'P_cond_MPa', 'ideal_dryness_xs']].tail())
        
        # (可选) 保存包含新计算列的结果到新文件
        output_filename = 'fulldata_with_dryness.xlsx'
        print(f"\n正在将完整结果保存到: {output_filename}...")
        result_df.to_excel(output_filename, index=False)
        print("保存成功！")

正在读取文件: minute_Part234678-fulldata.xlsx...
文件读取成功！
正在准备数据...
使用大气压力 101.325 kPa 将真空度转换为绝对压力...
正在逐行计算理想排汽干度，这可能需要一些时间...
计算完成！

--- 计算结果摘要 ---
           P_ms_MPa        T_ms_C    P_cond_MPa  ideal_dryness_xs
count  21518.000000  21518.000000  21518.000000      21518.000000
mean      12.759069    530.625741      0.197145          0.897748
std        0.912485      7.603769      0.001834          0.009019
min        1.308589    337.717385      0.188767          0.873703
25%       13.018485    528.064348      0.195907          0.892920
50%       13.162996    531.563444      0.197427          0.895096
75%       13.206603    534.563146      0.198505          0.898250
max       13.716359    554.638333      0.201093          1.031770

总行数: 21518
成功计算行数: 21518
计算失败/跳过行数: 0
过热蒸汽工况数量 (干度 > 1): 5 (0.02%)

--- 结果预览 (前5行) ---
    P_ms_MPa      T_ms_C  P_cond_MPa  ideal_dryness_xs
0   3.718636  491.516667    0.192072          0.993450
1   8.846120  497.838556    0.198595          0.916743
2   9.8393

In [ ]:
def processed_plant_data(filepath, atm_pressure_kpa=101.325):
    df = pd.read_excel(filepath)
    # 计算均值，并处理可能只有单列存在的情况
    df['P_ms_MPa'] = df['pf_pt2502']
    df['T_ms_C'] = df['pf_rdlc04']
    # 计算真空度均值
    vacuum_kpa = df['of_pt2807']
    # vacuum_kpa=vacuum_kpa*(1-0.158)
    # --- 2. 关键步骤：将真空度转换为绝对压力 ---
    # 公式: 绝对压力(kPa) = 大气压(kPa) - 真空度(kPa)
    #       绝对压力(MPa) = (大气压(kPa) - 真空度(kPa)) / 1000
    print(f"使用大气压力 {atm_pressure_kpa} kPa 将真空度转换为绝对压力...")
    df['P_cond_MPa'] = (atm_pressure_kpa + vacuum_kpa) / 1000
    # 创建一个新的列 'ideal_dryness_xs'
    # 使用 .loc 避免 SettingWithCopyWarning
    df.loc[:, 'ideal_dryness_xs'] = df.apply(
        lambda row: calculate_ideal_dryness(row['P_ms_MPa'], row['T_ms_C'], row['P_cond_MPa']),
        axis=1
    )
    print("\n--- 计算结果摘要 ---")
    print(df[['P_ms_MPa', 'T_ms_C', 'P_cond_MPa', 'ideal_dryness_xs']].describe())
    return df

In [16]:
print(result_df['ideal_dryness_xs'].describe() ) # 显示理想干度的统计信息
print('________________')
processed_plant_data('coaldata_up_cluster_1.xlsx', atm_pressure_kpa=101.325)
processed_plant_data('coaldata_up_cluster_2.xlsx', atm_pressure_kpa=101.325)
processed_plant_data('coaldata_down_cluster_1.xlsx', atm_pressure_kpa=101.325)
processed_plant_data('coaldata_down_cluster_2.xlsx', atm_pressure_kpa=101.325)

count    21518.000000
mean         0.897748
std          0.009019
min          0.873703
25%          0.892920
50%          0.895096
75%          0.898250
max          1.031770
Name: ideal_dryness_xs, dtype: float64
________________
正在读取文件: coaldata_up_cluster_1.xlsx...
文件读取成功！
正在准备数据...
使用大气压力 101.325 kPa 将真空度转换为绝对压力...
正在逐行计算理想排汽干度，这可能需要一些时间...
计算完成！

--- 计算结果摘要 ---
          P_ms_MPa       T_ms_C   P_cond_MPa  ideal_dryness_xs
count  1408.000000  1408.000000  1408.000000       1408.000000
mean     13.107172   529.932134     0.181201          0.890285
std       0.268680     5.621812     0.001210          0.004054
min      10.426417   509.257583     0.178462          0.877163
25%      13.124140   526.951540     0.180254          0.888130
50%      13.174045   530.459512     0.181235          0.890176
75%      13.214790   533.740409     0.182224          0.892182
max      13.423667   550.028900     0.183578          0.915471

总行数: 1408
成功计算行数: 1408
计算失败/跳过行数: 0
过热蒸汽工况数量 (干度 > 1): 0 (0.00

,id,time,mw,uf_te1501,pf_at1601,pf_at1602,pf_ft2902,pf_pt2502,pf_rdlc04,pf_te2512,...,of_ft2903,of_m1501,of_m1602,of_m1702,of_pt2807,of_x12nao,P_ms_MPa,T_ms_C,P_cond_MPa,ideal_dryness_xs
0,1,0.000000,130.711273,16.332731,2.822657,3.886329,347.286633,13.151176,537.212300,551.86850,...,401.942531,61.526553,153.891821,73.796148,97.117204,347.208972,13.151176,537.212300,0.183098,0.894666
1,1,0.256757,130.671300,16.398881,3.154131,4.039014,347.728520,13.142189,533.797063,550.87275,...,410.349190,61.483343,153.666208,73.612018,97.118973,345.914218,13.142189,533.797063,0.183099,0.892751
2,1,2.481982,130.274200,16.348163,2.741598,3.798750,348.700188,13.139051,534.261500,547.57700,...,399.149485,61.465709,154.500125,73.894461,97.097879,347.265268,13.139051,534.261500,0.183081,0.893043
3,1,3.423423,130.106333,16.071075,2.381766,3.614840,348.197440,13.088575,529.092917,545.82625,...,450.046830,61.561339,154.262495,73.200503,97.121867,347.302731,13.088575,529.092917,0.183102,0.890427
4,1,6.076577,128.220357,15.839160,2.699027,3.818921,350.750061,13.181552,530.282300,552.31450,...,396.020602,61.533275,154.734452,73.365889,97.206712,348.511629,13.181552,530.282300,0.183173,0.890389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578,46,11.978261,108.718417,7.477636,3.837549,4.965260,290.215767,13.148167,528.926571,543.98900,...,372.804148,54.529050,144.021300,66.457690,99.598705,213.272621,13.148167,528.926571,0.185187,0.890364
579,46,15.489130,104.498786,8.174517,4.121661,5.365960,284.275900,13.156283,525.640750,546.09425,...,358.575850,54.771409,144.908779,66.906811,99.588023,246.549346,13.156283,525.640750,0.185178,0.888345
580,46,15.695652,104.507900,8.280415,4.076254,5.378718,283.879690,13.189931,526.009556,546.08300,...,353.118862,54.795746,144.688869,65.591920,99.584950,244.710490,13.189931,526.009556,0.185176,0.888290
581,46,16.108696,103.814454,8.379707,4.376291,5.434142,278.500250,13.179062,526.707400,546.91514,...,353.557446,54.641926,143.555750,65.955942,99.564799,249.992521,13.179062,526.707400,0.185159,0.888789


In [27]:
print(calculate_ideal_dryness(P_ms=10.664353, T_ms=501.416714, P_cond=(101.325+99.5)/1000))
print(calculate_ideal_dryness(P_ms=3.664353, T_ms=300.416714, P_cond=(101.325+80.5)/1000))

for p_cond in np.arange((101.325+99.5*(1-0.05))/1000, (101.325+99.5)/1000, 0.001):
    x_s = calculate_ideal_dryness(P_ms=3.664353, T_ms=300.416714, P_cond=p_cond)
    print(f"P_cond={p_cond:.4f} MPa, ideal_dryness_xs={x_s:.4f}")

print("----")

for p_cond in np.linspace((101.325+99.5*(1-0.9))/1000, (101.325+99.5)/1000, 101):
    x_s = calculate_ideal_dryness(P_ms=13, T_ms=500,P_cond=p_cond)
    print(f"P_cond={p_cond:.4f} MPa, ideal_dryness_xs={x_s:.4f}")



0.9000270359993785
0.8698270149124071
P_cond=0.1958 MPa, ideal_dryness_xs=0.8731
P_cond=0.1968 MPa, ideal_dryness_xs=0.8733
P_cond=0.1978 MPa, ideal_dryness_xs=0.8735
P_cond=0.1988 MPa, ideal_dryness_xs=0.8738
P_cond=0.1998 MPa, ideal_dryness_xs=0.8740
----
P_cond=0.1113 MPa, ideal_dryness_xs=0.8526
P_cond=0.1122 MPa, ideal_dryness_xs=0.8529
P_cond=0.1131 MPa, ideal_dryness_xs=0.8532
P_cond=0.1140 MPa, ideal_dryness_xs=0.8535
P_cond=0.1149 MPa, ideal_dryness_xs=0.8538
P_cond=0.1158 MPa, ideal_dryness_xs=0.8542
P_cond=0.1166 MPa, ideal_dryness_xs=0.8545
P_cond=0.1175 MPa, ideal_dryness_xs=0.8548
P_cond=0.1184 MPa, ideal_dryness_xs=0.8551
P_cond=0.1193 MPa, ideal_dryness_xs=0.8554
P_cond=0.1202 MPa, ideal_dryness_xs=0.8557
P_cond=0.1211 MPa, ideal_dryness_xs=0.8560
P_cond=0.1220 MPa, ideal_dryness_xs=0.8563
P_cond=0.1229 MPa, ideal_dryness_xs=0.8566
P_cond=0.1238 MPa, ideal_dryness_xs=0.8569
P_cond=0.1247 MPa, ideal_dryness_xs=0.8572
P_cond=0.1256 MPa, ideal_dryness_xs=0.8575
P_cond=0.12

In [ ]:
P_ms=10
T_ms=500
P_cond=0.2
T_ms_K = T_ms + 273.15
inlet_steam = IAPWS97(P=P_ms, T=T_ms_K)
s_ms = inlet_steam.s
        

outlet_sat_liquid = IAPWS97(P=P_cond, x=0)
s_f = outlet_sat_liquid.s
        
outlet_sat_vapor = IAPWS97(P=P_cond, x=1)
s_g = outlet_sat_vapor.s
x_s = (s_ms - s_f) / (s_g - s_f)
print(s_ms, s_f, s_g,x_s)
IAPWS97()

6.599322534689143 1.5300982011075013 7.126856391468609 0.905742960686063
